# 0. Mon setup colab

In [ ]:
# Adapte des TP.
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = True  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "pandas>=2.2,<4",
        "plotly>=6,<7",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")


Installation finished. Restart the session if Colab requests it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/ammou35/LNH-data-science.git

Cloning into 'LNH-data-science'...
remote: Enumerating objects: 491, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 491 (delta 160), reused 179 (delta 127), pack-reused 259 (from 1)
Receiving objects: 100% (491/491), 1.58 MiB | 7.19 MiB/s, done.
Resolving deltas: 100% (253/253), done.


In [ ]:
%cd /content/LNH-data-science/

/content/LNH-data-science


In [ ]:
!git switch advanced_viz

branch 'advanced_viz' set up to track 'origin/advanced_viz'.
Switched to a new branch 'advanced_viz'


In [ ]:
!git pull

Already up to date.


In [ ]:
%cd data_science_project/

/content/LNH-data-science/data_science_project


In [ ]:
!uv sync

Using CPython 3.11.17
Creating virtual environment at: .venv
Resolved 204 packages in 2ms
Prepared 129 packages in 9.05s
Installed 129 packages in 488ms
 + annotated-types==0.8.0
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + arrow==1.4.0
 + astor==0.8.1
 + asttokens==3.0.2
 + async-lru==2.3.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.22
 + decorator==5.3.1
 + defusedxml==0.7.1
 + duckdb==1.5.5
 + et-xmlfile==2.0.0
 + executing==2.2.1
 + fastjsonschema==2.22.2
 + fonttools==4.65.0
 + fqdn==1.5.1
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + ift6758==0.1.0 (from file:///content/LNH-data-science/data_science_project)
 + iniconfig==2.3.1
 + ipdb==0.13.13
 + ipykernel==7.3.0
 + ipython==9.17.1
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.9
 + isoduration==20.11.0
 +

In [ ]:
# Setup
import sys, ift6758
print(sys.executable)
print(ift6758.__file__)

from ift6758.data import save_to_csv

import json
import os
from pathlib import Path
import numpy as np
import pandas as pd

from IPython.display import display

# Racine du projet = le dossier data_science_project (ce notebook est dans son sous-dossier notebooks/).
# Sert à trouver le cache de données (data/raw) et l'image de la patinoire (figures/nhl_rink.png).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "data_science_project")
os.environ["NHL_DATA_DIR"] = str(ROOT / "ift6758/data")
print("Data directory:", os.environ["NHL_DATA_DIR"])

from ift6758.data import get_game
from ift6758.data.nhl import GAME_TYPE_CODE, get_data_dir

/usr/bin/python3
/content/LNH-data-science/data_science_project/ift6758/__init__.py
Data directory: /content/LNH-data-science/data_science_project/ift6758/data


In [ ]:
os.environ['NHL_DATA_DIR'] = '/content/drive/MyDrive/ift6758/data/'
data_dir = Path(os.environ['NHL_DATA_DIR'])
raw_data_dir = data_dir / "raw"
processed_data_dir = data_dir / "processed"

print(raw_data_dir)
print(processed_data_dir)

/content/drive/MyDrive/ift6758/data/raw
/content/drive/MyDrive/ift6758/data/processed


# 5. Visualisations avancées : cartes de tirs

## Consignes

Le dernier ensemble de visualisations que vous créerez sont des cartes de tirs pour une équipe donnée de la LNH, pour une année et une saison données. Un excellent exemple de ces graphiques, avec une description détaillée de comment les lire, se trouve sur le [site web de hockeyviz](https://hockeyviz.com/howto/shotMap) (qui est une excellente ressource pour beaucoup de choses en science des données appliquée au hockey). Notez que vous devrez créer ces figures à partir de zéro ; pour cette étape, vous ne pouvez utiliser aucune bibliothèque qui génère des figures spécifiques au domaine (hockey) pour vous. Vous recevrez un exemple d'image de patinoire ayant le bon ratio.

Pour créer ces figures, vous devez :

1. Vous assurer que vous pouvez travailler correctement avec les coordonnées des événements. Cela inclut de vous assurer que les tirs sont du bon côté de la patinoire (en raison des changements de période, ou parce que les équipes commencent de côtés différents pendant un match), ainsi que de pouvoir convertir les coordonnées physiques en coordonnées de pixels sur la figure.  
2. Calculer des statistiques agrégées des emplacements de tirs pour l'ensemble de la ligue afin d'obtenir le *taux de tirs moyen par heure de la ligue*. Pour chaque zone (*bin*) spatiale, divisez le nombre de tirs de toutes les équipes par `2 * nombre de matchs de la ligue` pour cette saison : chaque match contribue deux heures-équipe selon l'hypothèse de 60 minutes. Vous pouvez faire quelques hypothèses simplificatrices :  
   - Vous pouvez supposer que tous les tirs sont à forces égales ; cela signifie que vous pouvez simplement agréger tous les tirs plutôt que de déterminer si un tir a été effectué à forces égales ou non (rappel : Q 4.2).  
   - Vous pouvez supposer que chaque match dure 60 minutes.  
3. Regrouper les tirs par équipe et utiliser le *taux de tirs moyen par heure* de la ligue calculé ci-dessus pour calculer le *taux de tirs excédentaire par heure*. Pour chaque zone, divisez le nombre de tirs de l'équipe par son nombre de matchs dans cette saison, puis soustrayez le taux de la ligue pour cette zone. Rapportez la différence brute en **tirs par heure-équipe**.  
4. Faire des choix appropriés pour regrouper (*binning*) vos données lors de leur affichage. Vous pouvez également envisager d'utiliser des techniques de lissage pour rendre vos cartes de tirs plus lisibles. Une stratégie courante consiste à utiliser l'estimation par noyau (*kernel density estimation*) avec un noyau gaussien.  
5. Rendre le graphique interactif, avec des options pour sélectionner l'**équipe**. La façon la plus simple de le faire est d'utiliser un outil comme plotly ou bokeh. Une petite démo simple de ce que vous pouvez faire avec plotly se trouve [ici](https://plotly.com/python/dropdowns/).  
6. Produire **une figure interactive pour chaque saison de 2016-17 à 2020-21** (inclusivement). Vous devez **seulement créer les figures de la zone offensive** ; vous n'avez rien à faire pour la zone défensive. Comme mentionné ci-dessus, vous pouvez aussi ignorer si un événement s'est produit lors d'un avantage ou d'un désavantage numérique.

[![Exemple de carte de tirs offensive de hockeyviz](images/hockeyviz-shot-map.png)](https://hockeyviz.com/howto/shotMap)  
*Source : [www.hockeyviz.com](http://www.hockeyviz.com). Exemple de carte de tirs offensive pour les Sharks de San Jose, pour la saison 2017-2018. Vous **n'avez pas** besoin de calculer le xGF/60 ni le nombre de minutes passées en zone offensive.*

**Questions**

1. [Exportez les 5 graphiques de zone offensive en HTML](https://plotly.com/python/interactive-html-export/) et **intégrez-les dans votre article de blog**. Votre graphique doit permettre aux utilisateurs de sélectionner n'importe quelle équipe pour la saison sélectionnée.  
   ***Note** : Puisque vous pouvez trouver ces figures sur Internet, répondre à ces questions sans produire ces figures* ne *vous rapportera aucun point\!*  
     
2. **Discutez** (en quelques phrases) de ce que vous pouvez interpréter à partir de ces graphiques.  
     
3. Considérez l'Avalanche du Colorado ; regardez leur carte de tirs pendant la saison 2016-17. **Discutez** de ce que vous pourriez dire sur l'équipe pendant cette saison. Regardez maintenant la carte de tirs de l'Avalanche du Colorado pour la saison 2020-21, et **discutez** de ce que vous pourriez conclure de ces différences. Est-ce que cela a du sens? *Indice : regardez le classement*.  
     
4. Considérez les Sabres de Buffalo, une équipe qui a connu des difficultés ces dernières années, et comparez-les au Lightning de Tampa Bay, une équipe qui a remporté la coupe Stanley deux années de suite. Regardez les cartes de tirs de ces deux équipes pour les saisons 2018-19, 2019-20 et 2020-21. **Discutez** des observations que vous pouvez faire. Y a-t-il quelque chose qui pourrait expliquer le succès du Lightning, ou les difficultés des Sabres? Dans quelle mesure pensez-vous que ces figures donnent un portrait complet?

***Remarque** : le but de cet exercice est de vous familiariser avec l'utilisation des bibliothèques Python standards pour créer des visualisations. Vous ne pouvez pas utiliser d'outil qui crée pour vous des visualisations spécifiques au domaine (c.-à-d. le hockey). Vous êtes libres d'utiliser des bibliothèques standards (matplotlib, seaborn, plotly, bokeh, etc.) pour générer ces graphiques.*


## Definition des fonctions utilitaires

In [ ]:
def import_processed_data(
        data_dir: str | os.PathLike | None = os.environ['NHL_DATA_DIR'],
        start_season: int = 2016,
        end_season: int = 2023,
        export: bool = False,
        verbose: bool = False):
    """
    Fonction utilitaire pour importer les donnees tidy deja creees, ou bien
    traiter les donnees brutes telechargees lorsque les donnees tidy n'existent
    pas.

    Parametres:
        data_dir: le chemin du dossier data/ a la racine des dossier raw/ et processed/.
        start_season: saison de depart a chercher.
        end_season: saison de fin a chercher.
        export: sauvegarder ou non les donnees tidy lorsqu'elles n'existaient pas.
        verbose: afficher ou non les messages d'information.

    Retourne:
        DataFrame Pandas des donnees tidy importees ou produites.
    """

    processed_data_dir = Path(data_dir) / "processed"

    # On cree les dataframes pour chaque saison
    dataframes = []

    for season in range(start_season, end_season+1):
        filepath = processed_data_dir / f"shots_{season}.csv"
        if verbose:
            print(f"Fichier en cours: {filepath}")

        if (filepath.is_file()):
            df = pd.read_csv(filepath)
            if verbose:
                print(f"Le fichier de la saison {season} existe deja, on passe a la saison suivante\n")
        else:
            try:
                if verbose:
                    print(f"Creation du tidy df de la saison {season}...")
                df = season_to_pandas(season, data_dir=Path(data_dir) / "raw")
            except:
                print("Erreur")

            if export:
                if verbose:
                    print(f"Sauvegarde de shots_{season}...")
                save_to_csv(df, f"shots_{season}", processed_data_dir)

            if verbose:
                print(f"Saison {season} faite")

        dataframes.append(df)

    if verbose: print("C'est tout bon!")

    # On regroupe tout dans un seul dataframe
    return pd.concat(dataframes, ignore_index=True)


def infer_attack_direction(dataframe, normalized=True):
    """Infere la direction de l'equipe qui attaque:
    positif quand l'equipe attaque vers la droite, negatif autrement.

    Se base sur une heuristique du nombre majoritaire de label de zone O pour
    une periode de jeu donnee.
    En donnant un score de 1 au shots avec le label de zone O et et -1 aux
    autres, alors le signe de (x * zone_score) donne la direction.
    En effet, quand une equipe attaque vers la droite, la zone O est celle qui
    correspond a la droite de la patinoire, et vice versa.

    Parametres:
        dataframe: DataFrame Pandas; necessite les colonnes 'season', 'game_id',
        'period', 'team_id', 'zone' et 'x'.
        normalized: les valeurs retournees sont egales a -1 ou 1 si vrai,
        autrement c'est le score de l'heuristique qui est retourne.

    Retourne:
        Series qui contient une colonne 'attack_dir' qui donne la direction
        de l'equipe qui attaque.
    """

    if normalized is True:
        func = np.sign
    else:
        def func(x): return x

    return (dataframe
            .assign(zone_score=lambda df: (df['zone'] == 'O')*2 - 1,
                    attack_score=lambda df: (df['x']*df['zone_score'] > 0)*2 - 1)
            .groupby(['season', 'game_id', 'period', 'team_id'])
            ['attack_score'].transform('sum')
            .transform(lambda s: func(s))
            )

def compute_distance_to_net(
    dataframe,
    xcol: str = 'x',
    ycol: str = 'y',
    dircol: str = 'attack_direction'):
    """Calcule la distance au filet adverse d'ou le tir est effectue.

    Les deux premieres colonnes sont considerees contenir 'x' et 'y' si l'entree
    ne contient pas de colonnes nommees 'x' et 'y'.
    Si elle existe, la troisieme colonne est consideree contenir la direction
    vers laquelle le tir est effectue (vers la gauche si valeur negative, vers
    la droite si valeur positive). Sinon, la direction est inferee par la
    fonction infer_attack_direction.

    Parametres:
        df: DataFrame Pandas contenant les coordonnees pour le calcul de la
        distance.
        xcol: le nom de la colonne contenant les valeurs de x.
        ycol: le nom de la colonne contenant les valeurs de y.
        dircol: le nom de la colonne contenant les valeurs de la direction
        vers laquelle le tir est effectue.
    Retourne:
        Une Series Pandas contenant la distance au filet adverse.
    """

    # On verifie si le dataframe contient les colonnes nommees
    #columns = dataframe.columns
    #argcols = {
    #    "xcol": xcol,
    #    "ycol": ycol,
    #    "dircol": dircol
    #}


    #for idx, (colkey, colname) in enumerate(argcols.items()):
    #    if colname not in columns:
    #        argcols[colkey] = idx


    # S'il n'y a pas de troisieme colonne, on infere la direction de l'attaque
    #if

    # TODO: terminer l'implementation complete
    # Fonctionne pour la base, pas pour le fancy

    return dataframe.assign(d=lambda df: np.sqrt((89 - df[xcol]*np.sign(df[dircol]))**2 + df[ycol]**2))['d']

In [ ]:
combined_shots = import_processed_data(verbose=True)

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2016.csv
Le fichier de la saison 2016 existe deja, on passe a la saison suivante

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2017.csv
Le fichier de la saison 2017 existe deja, on passe a la saison suivante

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2018.csv
Le fichier de la saison 2018 existe deja, on passe a la saison suivante

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2019.csv
Le fichier de la saison 2019 existe deja, on passe a la saison suivante

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2020.csv
Le fichier de la saison 2020 existe deja, on passe a la saison suivante

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2021.csv
Le fichier de la saison 2021 existe deja, on passe a la saison suivante

Fichier en cours: /content/drive/MyDrive/ift6758/data/processed/shots_2022.csv
Le fichie

In [ ]:
combined_shots.columns

Index(['season', 'game_id', 'event_id', 'game_type', 'period', 'period_type',
       'time_in_period', 'team', 'team_id', 'is_home_team', 'event_type',
       'is_goal', 'x', 'y', 'player_name', 'goalie_name', 'shot_type', 'zone',
       'empty_net', 'away_goalie_in_net', 'away_skaters', 'home_skaters',
       'home_goalie_in_net', 'strength'],
      dtype='object')

## 1.

"Vous assurer que vous pouvez travailler correctement avec les coordonnées des événements. Cela inclut de vous assurer que les tirs sont du bon côté de la patinoire (en raison des changements de période, ou parce que les équipes commencent de côtés différents pendant un match), ainsi que de pouvoir convertir les coordonnées physiques en coordonnées de pixels sur la figure."

On va dire oui pour le moment

In [ ]:
# Inclure les SO?
combined_shots.groupby(['season', 'period'])['period_type'].value_counts()

season  period  period_type
2016    1       REG            25780
        2       REG            27377
        3       REG            25122
        4       OT              1480
        5       SO               601
                OT                39
2017    1       REG            28096
        2       REG            29479
        3       REG            27488
        4       OT              1406
        5       SO               635
                OT                33
2018    1       REG            27922
        2       REG            29063
        3       REG            27153
        4       OT              1249
        5       SO               524
                OT                28
2019    1       REG            23970
        2       REG            25388
        3       REG            23221
        4       OT              1254
        5       SO               523
                OT                56
        6       OT                22
        7       OT                22
        8       OT                 4
2020    1       REG            18496
        2       REG            19835
        3       REG            17970
        4       OT               996
        5       SO               375
                OT                56
        6       OT                 6
2021    1       REG            28737
        2       REG            30505
        3       REG            28366
        4       OT              1319
        5       SO               609
                OT                36
        6       OT                13
2022    1       REG            28132
        2       REG            30253
        3       REG            27643
        4       OT              1427
        5       SO               548
                OT                41
        6       OT                19
        7       OT                23
2023    1       REG            27129
        2       REG            29056
        3       REG            27061
        4       OT              1169
        5       SO               498
                OT                19
Name: count, dtype: int64

In [ ]:
# On drop les valeurs NA (maintenant qu'on sait combien on en a pour chaque annees) (pris du TP)
# "Excluez les valeurs manquantes seulement des analyses qui en ont besoin, et rapportez le nombre d'événements exclus."
combined_shots_with_coordinates = combined_shots.dropna(subset=['x', 'y'])
print("Rows excluded from a coordinate-based analysis:", len(combined_shots) - len(combined_shots_with_coordinates))
print("Rows kept in the full shots table:", len(combined_shots))

Rows excluded from a coordinate-based analysis: 17
Rows kept in the full shots table: 648272


Assignons les distances et la direction de l'attaque pour chaque tir.

In [ ]:
combined_shots_with_coordinates = (combined_shots_with_coordinates
                                   .assign(attack_direction=infer_attack_direction,
                                           distance=compute_distance_to_net)
                                   )

In [ ]:
distance_features = ['event_id', 'period', 'period_type', 'team', 'is_home_team', 'zone', 'x', 'y', 'distance', 'attack_direction']

In [ ]:
combined_shots_with_coordinates[distances_features]

,event_id,period,period_type,team,is_home_team,zone,x,y,distance,attack_direction
0,8,1,REG,TOR,False,O,-77.0,5.0,13.000000,-1
1,11,1,REG,OTT,True,O,86.0,13.0,13.341664,1
2,15,1,REG,OTT,True,N,23.0,-38.0,76.157731,1
3,16,1,REG,OTT,True,O,33.0,-15.0,57.974132,1
4,24,1,REG,TOR,False,O,-34.0,28.0,61.717096,-1
...,...,...,...,...,...,...,...,...,...,...
648267,252,3,REG,EDM,False,O,85.0,1.0,4.123106,1
648268,1183,3,REG,FLA,True,O,-52.0,-27.0,45.803930,-1
648269,1189,3,REG,FLA,True,O,-59.0,-29.0,41.725292,-1
648270,1193,3,REG,EDM,False,O,57.0,-26.0,41.231056,1


In [ ]:
# Quelques verifs rapides: en zone O la distance est toujours inferieure a sqrt((89-25)**2 + (85/2)**2),
# en zone D toujours sup a sqrt((89+26)**2 + (85/2)**2), en zone N entre les deux
{'max_O': np.sqrt((89-25)**2 + (85/2)**2), 'min_D': np.sqrt((89+26)**2 + (85/2)**2)}

{'max_O': np.float64(76.8261023350788),
 'min_D': np.float64(122.60199835239229)}

In [ ]:
combined_shots_with_coordinates[['zone', 'distance']].query('zone != "SO"').groupby('zone').describe()

distance                                                            \
         count        mean        std        min         25%         50%   
zone                                                                       
D      11859.0  151.064331  27.781117   3.605551  136.108412  154.275727   
N      21384.0   85.142550  12.037492  63.000000   75.958870   83.934498   
O     615012.0   31.829818  17.427233   0.000000   15.811388   31.000000   

                              
             75%         max  
zone                          
D     170.293864  189.654950  
N      92.703829  121.433109  
O      46.043458  178.373204

On remarque des valeurs min et max anormales, cependant les autres valeurs statistiques sont coherantes. Il s'agit donc probablement des evenements mislabelled

Excluons-les pour voir la difference

In [ ]:
# On drop les valeurs pour qui ne remplissent pas le critere (attack_dir*x < 0) and (zone == "O") (adapte du TP)
combined_shots_cleaned = combined_shots_with_coordinates.query('~((attack_direction*x < 0) and (zone == "O"))')
print("Rows excluded from a coordinate-based analysis:", len(combined_shots_with_coordinates) - len(combined_shots_cleaned))
print("Rows kept in the full shots table:", len(combined_shots_cleaned))

Rows excluded from a coordinate-based analysis: 124
Rows kept in the full shots table: 648131


Verifions de nouveau les valeurs de distance

In [ ]:
combined_shots_cleaned[['zone', 'distance']].query('zone != "SO"').groupby('zone').describe()

distance                                                            \
         count        mean        std        min         25%         50%   
zone                                                                       
D      11859.0  151.064331  27.781117   3.605551  136.108412  154.275727   
N      21384.0   85.142550  12.037492  63.000000   75.958870   83.934498   
O     614888.0   31.806133  17.347475   0.000000   15.811388   31.000000   

                              
             75%         max  
zone                          
D     170.293864  189.654950  
N      92.703829  121.433109  
O      46.043458   97.169954

Les valeurs aberrantes ont disparu, mais il reste des anomalies! Entre autres, les distances des tirs dans la zone neutre doivent etre bornes par 89 -/+ 25.

On pourrait les trouver en cherchant pour quels shots en zone N la distance n'est pas dans les bornes.

De toute facon, nous ne nous interesserons qu'aux tirs effectues en O!

In [ ]:
data_dir

PosixPath('/content/drive/MyDrive/ift6758/data')

In [ ]:
# Sauvegarde du tableau avec les donnees de distance
save_to_csv(combined_shots_cleaned, "combined_shots_cleaned_with_distance", Path(data_dir / 'vizualisations/'))

Saved processed table: /content/drive/MyDrive/ift6758/data/vizualisations/combined_shots_cleaned_with_distance.csv


## 2.
"Calculer des statistiques agrégées des emplacements de tirs pour l'ensemble de la ligue afin d'obtenir le taux de tirs moyen par heure de la ligue. Pour chaque zone (bin) spatiale, divisez le nombre de tirs de toutes les équipes par 2 * nombre de matchs de la ligue pour cette saison : chaque match contribue deux heures-équipe selon l'hypothèse de 60 minutes. Vous pouvez faire quelques hypothèses simplificatrices :

    Vous pouvez supposer que tous les tirs sont à forces égales ; cela signifie que vous pouvez simplement agréger tous les tirs plutôt que de déterminer si un tir a été effectué à forces égales ou non (rappel : Q 4.2).
    Vous pouvez supposer que chaque match dure 60 minutes.
"

In [ ]:
# Il faut transformer x en -x seulenment lorsque la direction de l'attaque est vers la gauche afin d'avoir tous les tirs du meme cote de la patinoire pour l'analyse
# Autrement dit xbin = x*attack_dir, et ybin = -y
# On ne prend que les tirs en O !! -> .query('zone == "O"')
combined_shots_cleaned = (combined_shots_cleaned
                          .query('zone == "O"')
                          .assign(xbin=lambda df: df['x']*df['attack_direction'],
                                  ybin=lambda df: -df['y'])
                          )

In [ ]:
combined_shots_cleaned[['xbin', 'ybin']].describe()

,xbin,ybin
count,614888.000000,614888.000000
mean,62.817825,0.029145
std,16.759805,18.605572
min,-0.000000,-42.000000
25%,50.000000,-13.000000
50%,65.000000,-0.000000
75%,78.000000,14.000000
max,99.000000,42.000000


OK!

In [ ]:
# Adapte de https://stackoverflow.com/questions/43422961/2-dimensional-binning-with-pandas#43423056
xwidth = 20
ywidth = 10
combined_shots_cleaned["bins"] = list(zip(pd.cut(combined_shots_cleaned["xbin"], xwidth, labels=list(range(xwidth))),
                                                     pd.cut(combined_shots_cleaned["ybin"], ywidth, labels=list(range(ywidth)))))

In [ ]:
combined_shots_cleaned[distances_features + ['game_id', 'xbin', 'ybin']]

,event_id,period,period_type,team,is_home_team,zone,x,y,distance,attack_direction,game_id,xbin,ybin
0,8,1,REG,TOR,False,O,-77.0,5.0,13.000000,-1,2016020001,77.0,-5.0
1,11,1,REG,OTT,True,O,86.0,13.0,13.341664,1,2016020001,86.0,-13.0
3,16,1,REG,OTT,True,O,33.0,-15.0,57.974132,1,2016020001,33.0,15.0
4,24,1,REG,TOR,False,O,-34.0,28.0,61.717096,-1,2016020001,34.0,-28.0
5,25,1,REG,TOR,False,O,-33.0,-17.0,58.523500,-1,2016020001,33.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
648267,252,3,REG,EDM,False,O,85.0,1.0,4.123106,1,2023030417,85.0,-1.0
648268,1183,3,REG,FLA,True,O,-52.0,-27.0,45.803930,-1,2023030417,52.0,27.0
648269,1189,3,REG,FLA,True,O,-59.0,-29.0,41.725292,-1,2023030417,59.0,29.0
648270,1193,3,REG,EDM,False,O,57.0,-26.0,41.231056,1,2023030417,57.0,26.0


In [ ]:
# "Pour cette saison", "de toutes les equipes", "pour chaque bin"
shot_count_per_season_bin20x10 = combined_shots_cleaned.groupby(['season', 'bins']).size().rename('shots')
shot_count_per_season_bin20x10.sort_values(ascending=False)

season  bins   
2022    (16, 4)    5414
2023    (16, 4)    4762
2021    (16, 4)    4622
2022    (16, 5)    4168
2017    (15, 4)    4094
                   ... 
        (0, 8)        1
2016    (18, 9)       1
2017    (0, 9)        1
2016    (0, 1)        1
        (0, 0)        1
Name: shots, Length: 1226, dtype: int64

In [ ]:
# On aplati le groupby dataframe, puis on separe les coordonnes des bins
shot_count_per_season_bin20x10 = shot_count_per_season_bin20x10.reset_index()
shot_count_per_season_bin20x10[['x', 'y']] = shot_count_per_season_bin20x10['bins'].to_list()
shot_count_per_season_bin20x10

,season,bins,shots,x,y
0,2016,"(0, 0)",1,0,0
1,2016,"(0, 1)",1,0,1
2,2016,"(0, 3)",2,0,3
3,2016,"(0, 4)",8,0,4
4,2016,"(0, 5)",2,0,5
...,...,...,...,...,...
1221,2023,"(19, 4)",30,19,4
1222,2023,"(19, 5)",53,19,5
1223,2023,"(19, 6)",49,19,6
1224,2023,"(19, 7)",27,19,7


In [ ]:
# Nombre total de matchs
combined_shots_cleaned['game_id'].nunique()

10351

In [ ]:
# Nombre de matchs par saison: on regroupe par saison et on compte le nombre de game_id unique
games_per_season = combined_shots_with_coordinates.groupby('season')['game_id'].nunique().rename('games')
games_per_season

,games
season,
2016,1317
2017,1355
2018,1358
2019,1168
2020,952
2021,1401
2022,1400
2023,1400


In [ ]:
# On applique la formule qui nous est donnee
shot_count_per_season_bin20x10['2-hrs-team'] = shot_count_per_season_bin20x10.apply(lambda row: row['shots']/(2*games_per_season[row['season']]), axis=1)

In [ ]:
shot_count_per_season_bin20x10.sort_values(by='2-hrs-team',ascending=False)

,season,bins,shots,x,y,2-hrs-team
1040,2022,"(16, 4)",5414,16,4,1.933571
1192,2023,"(16, 4)",4762,16,4,1.700714
884,2021,"(16, 4)",4622,16,4,1.649536
269,2017,"(15, 4)",4094,15,4,1.510701
1041,2022,"(16, 5)",4168,16,5,1.488571
...,...,...,...,...,...,...
924,2022,"(0, 8)",1,0,8,0.000357
916,2021,"(19, 8)",1,19,8,0.000357
909,2021,"(18, 9)",1,18,9,0.000357
900,2021,"(18, 0)",1,18,0,0.000357


"Regrouper les tirs par équipe et utiliser le taux de tirs moyen par heure de la ligue calculé ci-dessus pour calculer le taux de tirs excédentaire par heure. Pour chaque zone, divisez le nombre de tirs de l'équipe par son nombre de matchs dans cette saison, puis soustrayez le taux de la ligue pour cette zone. Rapportez la différence brute en tirs par heure-équipe."

In [ ]:
# "Par equipe"
shot_count_per_season_team_bin20x10 = combined_shots_cleaned.groupby(['season', 'team', 'bins']).size().rename('count')
shot_count_per_season_team_bin20x10.sort_values(ascending=False)

season  team  bins   
2022    FLA   (16, 4)    266
2023    EDM   (16, 4)    251
2022    VGK   (16, 4)    241
        EDM   (16, 4)    236
        NJD   (16, 4)    228
                        ... 
2017    VAN   (19, 6)      1
        VGK   (0, 4)       1
              (5, 1)       1
2020    VGK   (19, 6)      1
        WPG   (5, 2)       1
Name: count, Length: 33483, dtype: int64

In [ ]:
#shot_count_per_season_team_bin20x10.groupby(['season', 'team']).describe()

count       mean        std  min   25%   50%    75%    max
season team                                                            
2016   ANA   175.0  17.382857  21.647828  1.0  3.00  11.0  25.00  131.0
       ARI   175.0  13.177143  15.080533  1.0  3.00   9.0  18.00   94.0
       BOS   169.0  17.201183  18.327568  1.0  3.00  12.0  26.00  109.0
       BUF   176.0  14.295455  16.732626  1.0  3.00  11.0  20.00  102.0
       CAR   157.0  16.012739  17.126598  1.0  3.00  13.0  21.00   93.0
...            ...        ...        ...  ...   ...   ...    ...    ...
2023   TOR   182.0  15.967033  20.262613  1.0  4.00   9.5  19.75  132.0
       VAN   188.0  13.851064  18.705690  1.0  3.00   8.0  18.00  138.0
       VGK   186.0  14.784946  17.255379  1.0  4.00   9.0  20.00  122.0
       WPG   188.0  14.000000  16.677501  1.0  3.75  10.0  17.25  119.0
       WSH   182.0  12.637363  15.386739  1.0  3.00   9.0  16.00   94.0

[250 rows x 8 columns]

In [ ]:
# On aplati le groupby dataframe, puis on sauvegarde nos donnees
shot_count_per_season_team_bin20x10 = shot_count_per_season_team_bin20x10.reset_index()
save_to_csv(shot_count_per_season_team_bin20x10, "nbin20x10", Path(data_dir / 'vizualisations/'), overwrite=True)

Saved processed table: /content/drive/MyDrive/ift6758/data/vizualisations/nbin20x10.csv


In [ ]:
games_per_season_team = combined_shots_with_coordinates.groupby(['season', 'team'])['game_id'].nunique().rename('games')
games_per_season_team

season  team
2016    ANA     99
        ARI     82
        BOS     88
        BUF     82
        CAR     82
                ..
2023    TOR     89
        VAN     95
        VGK     89
        WPG     87
        WSH     86
Name: games, Length: 250, dtype: int64

In [ ]:
# On peut transformer l'ancienne table pour fetch les valeurs de 2-hrs-team dessus plus facilement
lookup_2ht = shot_count_per_season_bin20x10.set_index(['season', 'bins'])['2-hrs-team']

In [ ]:
lookup_2ht

season  bins   
2016    (0, 1)     0.001898
        (0, 2)     0.001519
        (0, 3)     0.001898
        (0, 4)     0.001519
        (0, 6)     0.000759
                     ...   
2023    (19, 5)    0.100357
        (19, 6)    0.063929
        (19, 7)    0.037500
        (19, 8)    0.013929
        (19, 9)    0.000714
Name: 2-hrs-team, Length: 1587, dtype: float64

In [ ]:
# Il faudrait faire un pivot tout de suite?

In [ ]:
# On applique la formule donnee
shot_count_per_season_team_bin20x10['shot_per_team-hr'] = (shot_count_per_season_team_bin20x10
                                                           .apply(lambda row: row['count']/(games_per_season_team[(row['season'],row['team'])]) - lookup_2ht.loc[row['season'], row['bins']], axis=1)
                                                           )

In [ ]:
shot_count_per_season_team_bin20x10.sort_values(by='shot_per_team-hr', ascending=False)

,season,team,bins,count,shot_per_team-hr
24338,2021,VGK,"(16, 4)",195,0.728513
22067,2021,FLA,"(15, 4)",193,0.665992
26382,2022,FLA,"(16, 4)",266,0.648953
30645,2023,EDM,"(16, 4)",251,0.645080
21941,2021,EDM,"(16, 4)",220,0.595362
...,...,...,...,...,...
23805,2021,STL,"(16, 4)",101,-0.575068
11536,2018,TBL,"(16, 4)",63,-0.613171
32480,2023,SJS,"(16, 4)",88,-0.627544
3441,2016,TBL,"(16, 4)",50,-0.687510


In [ ]:
# On separe les coordonnes des bins et on exporte pour la suite
shot_count_per_season_team_bin20x10[['x', 'y']] = shot_count_per_season_team_bin20x10['bins'].to_list()
save_to_csv(shot_count_per_season_team_bin20x10.drop(columns=['count', 'bins']), "excess_shots_bin20x10", Path(data_dir / 'vizualisations/'), overwrite=True)

Saved processed table: /content/drive/MyDrive/ift6758/data/vizualisations/excess_shots_bin20x10.csv


## 3. Figures interactives

In [ ]:
# https://plotly.com/python-api-reference/generated/plotly.express.html#module-plotly.express

# Import required packages
import plotly.graph_objects as go
import numpy as np
import plotly.io as pio

In [ ]:
# Petite fonction utilitaire pour preparer le dataframe pour la figure a contour
def prepare_dataframe(df, season, team):
    # Pour une equipe et une saison donnee, pour afficher nos donnees, on peut les mettre dans une matrice de niveau (qui represente une sorte de heatmap), il faut donc pivoter la table
    return (df.query('season == @season and team == @team')
              .pivot_table(index="y", columns="x", values="shot_per_team-hr", aggfunc='sum')
              .to_numpy()
    )

In [ ]:
(df.query('season == @season and team == @team')
.pivot_table(index="y", columns="x", values="shot_per_team-hr", aggfunc='sum')
.to_numpy()
)

array([[            nan,             nan, -1.26714475e-03,
         1.31588108e-03,             nan,  2.05388846e-03,
                    nan,             nan,             nan,
        -4.01030425e-03, -1.21840841e-03,  5.26352433e-03,
                    nan, -7.19209079e-02, -2.12699297e-02,
        -3.09266866e-02, -6.75067883e-02, -5.57961429e-02,
        -2.25718861e-02,             nan],
       [ 5.37492167e-03,  2.08870013e-04,  1.07498433e-02,
         2.05388846e-03,             nan,  3.52990322e-03,
                    nan,  3.52990322e-03,             nan,
        -1.42727842e-03,  2.83993595e-02, -1.28176565e-02,
         6.75346376e-04, -9.67416278e-02, -3.87801991e-02,
        -1.03181787e-01,  9.76258442e-02, -6.53066908e-02,
        -4.75179280e-02,  1.00118360e-02],
       [            nan, -4.58817796e-03,  4.26791060e-03,
         1.31588108e-03,  1.31588108e-03,  2.08870013e-04,
         1.50177540e-02,  1.37018729e-02,  2.79189584e-03,
        -3.20267354e-04, -1.1

In [95]:
%cd ../data_science_blog

/content/LNH-data-science/data_science_blog


In [99]:
# Creation des figures avec Plotly (en suivant les indications dans le template du blog et les tutos sur le site officiel)

# Parametres de l'image de la patinoire
rinkpath = "./images/nhl_rink.png"


img_dims = {'x': 1100, 'y': 467}
half_rink_dims = {'x': 100, 'y': 85}
image_height_ft = half_rink_dims['x'] * img_dims['x'] / img_dims['y']

# Vecteurs pour le plot
nbinx = 20
nbiny = 10
x = np.linspace(0, half_rink_dims['x'], nbinx)   # Dimensions de la patinoire en ft
y = np.linspace(-half_rink_dims['y']/2, half_rink_dims['y']/2, nbiny)

# On fait un template pour le reutiliser selon l'annee choisie
# https://plotly.com/python/templates/#representing-themes-with-template-objects
excess_weighted_shots_template = go.layout.Template()

# Template du contour qu'on utilise pour plot les donnees de l'equipe choisie
excess_weighted_shots_template.data.contour = [
    go.Contour(
        x=x,
        y=y,
        ncontours=16,
        contours=dict(coloring="fill", showlines=True),
        connectgaps=True,
        colorscale="Turbo",
        opacity=0.45,
        visible=False
    )
]

excess_weighted_shots_template.layout = dict(
    autosize=False,
    #margin=dict(t=0, b=0, l=0, r=0),
    template="simple_white",
)

# L'image de la patinoire
excess_weighted_shots_template.layout.images = [dict(
    source=rinkpath,
    xref="x",
    yref="y",
    x=0,
    y=42,
    sizex=half_rink_dims['x'],
    sizey=image_height_ft,
    xanchor="center", # Pour n'avoir que la moitie droite de la patinoire
    yanchor="middle", # Pour etre centre en 0 sur la largeur
    #sizing="stretch",
    opacity=1,
    layer="below"
    )]

df=shot_count_per_season_team_bin20x10

# Boucle pour les 5 annees a montrer
for season in range(2016, 2021):
    teams = list(df.query('season == @season')['team'].unique())
    # create figure
    fig = go.Figure()
    fig.update_xaxes(range=[0, 100])   # Pour ne montrer que la moitie droite de la patinoire
    fig.update_yaxes(range=[-42, 42])  # Pour avoir le centre de la patinoire au centre du plot

    # Ajout du dropdown de selection de l'equipe pour la saison en question
    fig.update_layout(
        updatemenus=[
            dict(
                active=teams.index('SJS'),
                buttons=buttons,
            )
        ])


    # Boucle d'ajout de la trace pour chaque equipe pour une annee donnee
    # OU alors on peut utiliser add_traceS a la place

    #fig.add_traces({})
    buttons = list()
    for i, team in enumerate(teams):
        # Ajout de la trace
        fig.add_contour(z=prepare_dataframe(df, season, team))

        # Ajout du bouton correspondant dans la liste buttons
        # Pour les args: "visible" doit etre un vecteur one-hot pour la trace a afficher
        #                "title" = team
        #                "annotations" on verra
        buttons.append(dict(label=team,
                            method="update",
                            args=[{"visible": [False if j != i else True for j in range(len(teams))],
                                    "title": team,
                                    }]
                            ))


    fig.update_layout(template=excess_weighted_shots_template, title=dict(text="Taux de tirs excédentaire par heure\nSaison '%s'" % season))
    # De "plotly_html.ipynb" fourni dans le blog
    # Export the figure to an HTML snippet that can be included in a post.
    # include_plotlyjs="cdn" loads plotly.js from a CDN instead of embedding ~4MB of JavaScript in every figure.
    fig.write_html(f"./_includes/{season}_shot_map.html", include_plotlyjs="cdn", full_html=False)


fig.show()